# Results for model: nvidia/nemotron-4-mini-hindi-4b-instruct

In [ ]:
import polars as pl
import xgboost as xgb

# Set paths for data and model
data_path = './jane_street_data/train.parquet'
target_file = 'responder_6'

def features_from_dataframe(df, quantile):
    """
    Calculate a global TOP_QUANTILE (top 15%) of 'feature_00'
    relative to 'responder_6' across rolling batches of 'date_id'.

    Parameters:
    df (pandas.DataFrame): A pandas dataframe containing the raw data
    quantile (float): The fractional quantile to use (e.g. 0.15 = the top 15%)

    Returns:
    dict: A dictionary containing the updated 'feature_00' and responder_6 columns
    """
    # Generate rolling batches of 'date_id'
    # Supposed to start at 'year=2014' as that's where market data is available
    rolling_batches = pl.rolling(pl.pd.parquet_remote(data_path, 'date_id'))

    # Calculate the TOP_QUANTILE of 'feature_00'
    updated_df = df.to_pandas().values
    batch_stats = pl.rfunc.num_percentiles(
        updated_df[:, 0], batch_size=updated_df.shape[0], num_perc=quantile
    ).values())

    # Restore the response variable
    feature_colname = 'feature_00'
    responder_colname = 'responder_6'

    # Update the given 'feature_00' to be its TOP_QUANTILE (top 15%)
    new_feature_df = updated_df.transpose().apply(
        lambda x: x[feature_colname] if feature_colname in x else np.nan).update(
            batch_stats / updatable_size
    ).mean()

    return {responder_colname: df[responder_colname], 'feature_00': new_feature_df}


# Run features engineering process
dataset = pl.read_parquet_remote(data_path)
features = features_from_dataframe(dataset, quantile=0.15)

# Train an XGBoost Regressor
model = xgb.XGBRegressor(max_depth=4)
model.fit(features['feature_00'], features[target_file])